In [17]:
import numpy as np

In [18]:
class Tensor:

    def __init__(self, data, parents=(), creator=None, req_grad=False):
        self.data = np.asarray(data, dtype=np.float64)
        self.grad = np.zeros_like(self.data)
        self.parents = parents
        self.creator = creator
        self.requires_grad = req_grad

    def __repr__(self):
        op_name = "empty" if self.creator is None else f"{type(self.creator).__name__}"
        return f"Data={self.data}, grad={self.grad}, op={op_name}, requires_grad={self.requires_grad}"
    
    @property
    def shape(self):
        return self.data.shape
    
    def backward(self):
        if self.requires_grad == False:
            raise RuntimeError("No trainable parameter found set req_grad=True on your parameters")
        topo_arr = []
        self.topo_sort(set(),set(), topo_arr)
        topo_arr.reverse()
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            else:
                topo_arr[i].grad = np.zeros_like(topo_arr[i].data)
        self.grad = np.ones_like(self.data)
        for i in range(len(topo_arr)):
            if topo_arr[i].creator is None:
                continue
            if topo_arr[i].requires_grad == True:
                parent_gradients = topo_arr[i].creator.backward(topo_arr[i].grad)
                for n,s in zip(topo_arr[i].parents, parent_gradients):
                    if n.requires_grad == True:
                        if n.grad.shape != s.shape:
                            s = self.unbroadcast(n,s)
                        n.grad += s

    def unbroadcast(self, tgt_grad, act_grad):
        for i in range(len(act_grad.shape) - len(tgt_grad.shape)):
            act_grad = np.sum(act_grad, axis = 0)
        if len(tgt_grad.shape) != 0:
            for j in range(len(tgt_grad.shape)):
                if act_grad.shape[j] != tgt_grad.shape[j]:
                    act_grad = np.sum(act_grad, axis = j, keepdims=True)

        return act_grad    

    def zero_grad(self):
        self.grad = np.zeros_like(self.data)

    def topo_sort(self, visiting, visited, result):
        visiting.add(self)
        for i in range(len(self.parents)):
            if self.parents[i] not in visited:
                self.parents[i].topo_sort(visiting, visited, result)
        visiting.remove(self)
        visited.add(self)
        result.append(self)
        return


        



In [ ]:
import numpy as np


class operations:

    def forward(self, *args, **kwargs):
        raise NotImplementedError

    def backward(self, *args, **kwargs):
        raise NotImplementedError

    @classmethod
    def run(cls, *args, **kwargs):
        obj = cls()
        args = list(args)
        for i in range(len(args)):
            if not isinstance(args[i], Tensor):
                args[i] = Tensor(args[i])
                
        inputs = tuple(a.data for a in args)
        result = obj.forward(*inputs, **kwargs)
        obj.output = result
        for a in args:
            if a.requires_grad == True:
                req_grad = True
                break
            else:
                req_grad = False
        return Tensor(result, parents=args, creator=obj, req_grad=req_grad)

def grad_check(ops, *args):
    obj = ops()
    args = list(args)
    for i in range(len(args)):
        if not isinstance(args[i], Tensor):
            args[i] = Tensor(args[i])
    eps = Tensor(1e-6)
    # args = tuple(a.data for a in args)
    def nudged(arr,idx,delta):
        out = arr.copy()
        out[idx] += delta
        return out
    for i in range(len(args)):
        for idx in np.ndindex(args[i].data.shape):
            inputs = tuple(nudged(args[j].data, idx, eps.data) if j == i else args[j].data for j in range(len(args)))
            psv_res = obj.forward(*inputs)
            inputs = tuple(nudged(args[j].data, idx, -eps.data) if j == i else args[j].data for j in range(len(args)))
            neg_res = obj.forward(*inputs)
            grad = np.sum((psv_res - neg_res)/(2*eps.data))
            rel_error = abs(grad - args[i].grad[idx])/np.maximum(np.maximum(abs(grad),abs(args[i].grad[idx])),1e-12)
            if np.max(rel_error) >= 1e-5:
                return False
    return True

class Mul(operations):
    def forward(self, x, y):
        self.x,self.y = x,y
        return self.x*self.y
    def backward(self, gradient):
        return tuple([self.y*gradient, self.x*gradient])

In [70]:
# helpers (test-only)
class Dot(operations):                # output = ONE number from all elements
    def forward(self, x, y): self.x, self.y = x, y; return np.sum(x * y)
    def backward(self, g): return (self.y * g, self.x * g)

def chk(op, *vals):                   # backward first, then check
    ts = [Tensor(v, req_grad=True) for v in vals]
    op.run(*ts).backward()
    return grad_check(op, *ts)

# X2 — one input element feeds many outputs
# out = x * s = [10, 20, 30];  backward seeded with ones → measures sum(out) = 60
# d sum(out)/ds = 1 + 2 + 3 = 6   (shape ())
print("X2", chk(Mul, [1., 2., 3.], 10.))    # want True


9.999999999621423
[10. 10. 10.]
10.00000000139778
[10. 10. 10.]
10.000000003174137
[10. 10. 10.]
5.999999997285954
6.0
X2 True


In [137]:
x = Tensor([1., 2.]); W = Tensor([3., 4.], req_grad=True)
c = Mul.run(x, W); c.backward()
print(x.grad, W.grad, c.requires_grad)   # want: zeros, [1. 2.], True

p = Tensor([1.]); q = Tensor([2.])
print(Mul.run(p, q).requires_grad)      # want False

[0. 0.] [1. 2.] True
False


In [138]:
# A: intermediate node with no grad
p = Tensor([1.]); q = Tensor([2.]); W = Tensor([3.], req_grad=True)
r = Mul.run(p, q)
c = Mul.run(r, W); c.backward()
print(p.grad, r.grad, W.grad)      # want [0.] ? [2.]  ← what should r.grad be?

[0.] [0.] [2.]


In [143]:
# B: nothing trainable at all
p = Tensor([1.]); q = Tensor([2.])
c = Mul.run(p, q); c.backward()
print(c.grad)

RuntimeError: No trainable parameter found

In [144]:
def test(name, got, want):
    ok = np.allclose(got, want)
    print(("PASS " if ok else "FAIL ") + name, "| got", got, "| want", want)

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(a, 2); c.backward()
test("tensor*int", a.grad, [2., 2.])

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(3, a); c.backward()
test("int*tensor", a.grad, [3., 3.])

a = Tensor([2., 3.], req_grad=True)
c = Mul.run(a, 0.5); c.backward()
test("tensor*float", a.grad, [0.5, 0.5])

a = Tensor([[1., 2.], [3., 4.]], req_grad=True)
c = Mul.run(a, np.array([10., 20.])); c.backward()
test("tensor*ndarray", a.grad, [[10., 20.], [10., 20.]])

a = Tensor([1., 1.], req_grad=True)
c = Mul.run(Mul.run(a, 2), 2); c.backward()
test("chained const", a.grad, [4., 4.])


PASS tensor*int | got [2. 2.] | want [2.0, 2.0]
PASS int*tensor | got [3. 3.] | want [3.0, 3.0]
PASS tensor*float | got [0.5 0.5] | want [0.5, 0.5]
PASS tensor*ndarray | got [[10. 20.]
 [10. 20.]] | want [[10.0, 20.0], [10.0, 20.0]]
PASS chained const | got [4. 4.] | want [4.0, 4.0]
